# 9. Managed Identity + Microsoft Entra ID

This is a **very important production-security topic** for Azure AI applications.

## 1. What is Microsoft Entra ID?

**Microsoft Entra ID** is Microsoft's cloud identity and access management service.

It handles:

- Authentication — **Who are you?**
- Authorization — **What are you allowed to access?**
- Users
- Groups
- Applications
- Service principals
- Managed identities
- RBAC integration

In an Azure AI system:

```text
User / Application
        ↓
Microsoft Entra ID
        ↓
Authentication
        ↓
Authorization / RBAC
        ↓
Azure Resources
```

---

# 2. What is Managed Identity?

**Managed Identity** gives an Azure resource an identity in Microsoft Entra ID so that an application can authenticate to supported Azure services **without storing passwords, API keys, or client secrets in code**. 

Instead of:

```text
Python Application
      ↓
API Key / Password
      ↓
Azure Service
```

you use:

```text
Python Application
      ↓
Managed Identity
      ↓
Microsoft Entra ID
      ↓
Access Token
      ↓
Azure Service
```

---

# 3. Why Do We Need It?

### ❌ Without Managed Identity

```python
client = SomeAzureClient(
    endpoint="https://...",
    api_key="my-secret-key"
)
```

Problems:

- Secret can leak
- Secret rotation required
- Secret may accidentally enter Git
- Credential management overhead

### ✅ With Managed Identity

```python
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
```

Azure obtains the required token through the application's identity.

Microsoft recommends managed identities for Azure-to-Azure service authentication where supported. 

---

# 4. How They Work Together

This is the key concept:

> **Microsoft Entra ID is the identity platform. Managed Identity is a workload identity that Azure manages within Entra ID.**

Example:

```text
                 Microsoft Entra ID
                        │
               ┌────────┴────────┐
               │                 │
             Users         Managed Identities
                                 │
                    ┌────────────┼────────────┐
                    ▼            ▼            ▼
                 App Service   Function      VM
```

The managed identity can then be granted permissions to other Azure resources.

---

# 5. Simple Example

Suppose your AI application runs on:

```text
Azure App Service
```

and needs to access:

```text
Azure AI Search
Azure OpenAI
Blob Storage
Key Vault
```

Architecture:

```text
                 Azure App Service
                       │
                 Managed Identity
                       │
                       ▼
                Microsoft Entra ID
                       │
              Access Token
                       │
       ┌───────────────┼───────────────┐
       ▼               ▼               ▼
 Azure AI Search   Blob Storage     Key Vault
```

Permissions are granted using appropriate Azure roles/RBAC where supported.

---

# 6. Two Types of Managed Identity ⭐⭐⭐⭐⭐

There are two types:

1. **System-assigned**
2. **User-assigned** 


---

## 7. System-Assigned Managed Identity

Identity is created together with the Azure resource.

```text
App Service
    │
    └── System-Assigned Identity
```

Its lifecycle is tied to the resource.

If the resource is deleted, its system-assigned identity is also deleted. A system-assigned identity is associated with a single Azure resource. 

### Example

```text
App Service A
     ↓
System Identity A
     ↓
Azure AI Search
```

Good when:

> One application/resource needs its own identity.

---

# 8. User-Assigned Managed Identity

A user-assigned identity is created as a **separate Azure resource** and can be assigned to multiple Azure resources. Its lifecycle is independent of those resources. 

```text
              User-Assigned Identity
                    │
             ┌──────┼──────┐
             ▼      ▼      ▼
          App A   App B   Function
```

Good when:

- Multiple resources need the same identity
- You want identity lifecycle independent from compute
- Resources are frequently recreated
- You want consistent permissions

Microsoft's developer guidance recommends user-assigned managed identities for most scenarios where they are supported. 

---

# 9. System vs User Assigned

| | System Assigned | User Assigned |
|---|---|---|
| Created with | Azure resource | Separately |
| Lifecycle | Tied to resource | Independent |
| Can share? | ❌ No | ✅ Yes |
| Resource deleted | Identity deleted | Identity remains |
| Best for | Single workload | Multiple workloads |
| Reusable | ❌ | ✅ |
| Identity management | Simpler | More flexible |

### Interview answer

> "System-assigned identity is tied to the lifecycle of a single Azure resource, whereas user-assigned identity is an independent Azure resource that can be attached to multiple workloads."

---

# 10. How Authentication Actually Happens

Suppose:

```text
Azure Function
     ↓
Needs Blob Storage
```

The flow is:

```text
1. Function has Managed Identity
              ↓
2. Identity exists in Entra ID
              ↓
3. Function requests access token
              ↓
4. Entra ID issues token
              ↓
5. Function calls Blob Storage
              ↓
6. Blob validates token
              ↓
7. RBAC determines permission
              ↓
8. Access granted/denied
```

The important distinction:

```text
Entra ID
    ↓
Authentication
    ↓
"Who is this?"

RBAC
    ↓
Authorization
    ↓
"What can this identity do?"
```

---

# 11. RBAC ⭐⭐⭐⭐⭐

**Role-Based Access Control** determines what an identity can do.

For example:

```text
Managed Identity
       │
       ▼
Azure AI Search
       │
       ▼
Search-related Azure role
```

Or:

```text
Managed Identity
       │
       ▼
Storage Account
       │
       ▼
Appropriate Storage RBAC role
```

You should grant the **least privilege required**.

Don't simply give:

```text
Owner
```

to your AI application.

---

# 12. `DefaultAzureCredential`

This is very important for Python interviews.

```python
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
```

It provides a unified authentication approach across development and production environments.

Microsoft's Azure Identity library can use developer credentials locally and managed identity when the application runs in Azure, allowing the application code to remain largely unchanged. 

Conceptually:

```text
LOCAL
Python
 ↓
DefaultAzureCredential
 ↓
Developer Login / CLI / Environment


PRODUCTION
Python
 ↓
DefaultAzureCredential
 ↓
Managed Identity
 ↓
Microsoft Entra ID
```

---

# 13. Simple Python Example

Suppose your application needs Blob Storage.

```python
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient

credential = DefaultAzureCredential()

client = BlobServiceClient(
    account_url="https://mystorage.blob.core.windows.net",
    credential=credential
)

container = client.get_container_client("documents")

for blob in container.list_blobs():
    print(blob.name)
```

Notice:

```text
No API key
No password
No client secret
```

The credential is obtained through Azure Identity.

---

# 14. User-Assigned Identity in Python

If you use a user-assigned managed identity, you can explicitly specify its client ID:

```python
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential(
    managed_identity_client_id="YOUR_MANAGED_IDENTITY_CLIENT_ID"
)
```

Microsoft specifically documents using the user-assigned identity's **client ID** when configuring `DefaultAzureCredential`. 

---

# 15. Azure AI Search Example

Your production RAG application:

```text
                     App Service
                          │
                  Managed Identity
                          │
                          ▼
                   Microsoft Entra ID
                          │
                          ▼
                     RBAC Check
                          │
                          ▼
                  Azure AI Search
                          │
                          ▼
                    Search Results
```

Azure AI Search supports managed identities for outbound connections to supported Azure resources, using Entra security principals and role assignments. 

---

# 16. Agentic AI Example ⭐⭐⭐⭐⭐

This becomes especially important for your Altimetrik JD.

Suppose your agent has:

```text
HR Agent
 │
 ├── Azure OpenAI
 ├── Azure AI Search
 ├── Employee API
 ├── Blob Storage
 └── SQL Database
```

Don't give the agent a collection of API keys.

Instead:

```text
                    AI Agent
                       │
               Managed Identity
                       │
                       ▼
                Microsoft Entra ID
                       │
                RBAC / Permissions
                       │
        ┌──────────────┼──────────────┐
        ▼              ▼              ▼
   AI Search        Storage          SQL
```

This gives you a **credential-free service-to-service authentication pattern** for supported Azure resources.

---

# 17. Managed Identity + Key Vault

Suppose your application needs a third-party API key.

You could use:

```text
Application
     │
Managed Identity
     ↓
Microsoft Entra ID
     ↓
Key Vault
     ↓
Secret
     ↓
Application
```

So you don't put:

```python
OPENAI_API_KEY = "..."
```

inside the code.

Managed Identity authenticates the application to Key Vault, and Key Vault manages the secret.

---

# 18. Managed Identity vs API Key

| API Key | Managed Identity |
|---|---|
| Secret-based | Token-based |
| Must store secret | No application-managed secret |
| Rotation required | Azure manages identity credentials |
| Risk of leakage | Reduced secret exposure |
| Manual credential management | Azure-managed |
| Good for some external integrations | Preferred for Azure service-to-service where supported |

Managed identities eliminate the need for developers to manage the credentials used by the workload. 

---

# 19. Managed Identity vs Service Principal

This is a common interview question.

| Managed Identity | Service Principal |
|---|---|
| Azure-managed workload identity | Application identity in Entra ID |
| Credentials managed by Azure | Credentials/certificates traditionally managed by you |
| Designed for Azure resources/workloads | More general application identity |
| No secret management in application | May require secret/certificate management |
| Great for Azure-to-Azure | Useful for broader application scenarios |

Technically, managed identities are represented in Entra ID through a special type of service principal. 

---

# 20. User Authentication vs Application Authentication

Don't confuse these.

### User → Application

```text
Suraj
 ↓
Microsoft Entra ID
 ↓
Application
```

This answers:

> Who is the user?

### Application → Azure Service

```text
AI Application
 ↓
Managed Identity
 ↓
Microsoft Entra ID
 ↓
Azure AI Search
```

This answers:

> Which application/workload is making the request?

---

# 21. RAG Security Architecture

For your Azure RAG system:

```text
                         User
                           │
                           ▼
                    Microsoft Entra ID
                           │
                      Authentication
                           │
                           ▼
                       AI App
                           │
                  Managed Identity
                           │
                           ▼
                    Azure AI Search
                           │
                    RBAC / ACL Filter
                           │
                           ▼
                    Retrieved Context
                           │
                           ▼
                     Azure OpenAI
                           │
                           ▼
                       Response
```

This is a much better production architecture than:

```text
User
 ↓
API Key
 ↓
Search
 ↓
LLM
```

---

# 22. Important Distinction: Authentication vs Authorization

### Authentication

> **Who are you?**

Handled by identity systems such as Microsoft Entra ID.

### Authorization

> **What are you allowed to do?**

Handled through:

- Azure RBAC
- Application roles
- Resource permissions
- Data-level authorization
- API authorization

Example:

```text
Managed Identity
      ↓
Entra ID
      ↓
Authenticated
      ↓
RBAC
      ↓
Allowed?
   ├── Yes → Access
   └── No  → 403
```

---

# 23. Production AI Architecture

For your interview, connect this with everything we've studied:

```text
                         User
                           │
                           ▼
                   Microsoft Entra ID
                           │
                      Authentication
                           │
                           ▼
                     AI Application
                           │
                   Managed Identity
                           │
              ┌────────────┼─────────────┐
              ▼            ▼             ▼
       Azure AI Search   Blob         Key Vault
              │
              ▼
           RAG Data
              │
              ▼
        Azure OpenAI
              │
              ▼
       Content Safety
              │
              ▼
          Response
```

---

# 24. Interview Questions

### Q1. What is Managed Identity?

> "Managed Identity provides an Azure-managed workload identity that allows applications running on Azure to authenticate to supported Azure resources through Microsoft Entra ID without storing credentials such as API keys or passwords."

### Q2. What is Microsoft Entra ID?

> "Microsoft Entra ID is Microsoft's identity and access management platform. It provides authentication and integrates with authorization mechanisms such as Azure RBAC."

### Q3. What are the two types of Managed Identity?

> "System-assigned and user-assigned. System-assigned is tied to a single Azure resource's lifecycle, while user-assigned is an independent identity that can be attached to multiple resources."

### Q4. How does Managed Identity work?

> "The Azure workload gets an identity in Entra ID, requests an access token, and uses that token to access a downstream Azure resource. The target resource then evaluates the identity's permissions, typically through RBAC."

### Q5. Why use Managed Identity instead of API keys?

> "It removes the need to store and rotate application-managed credentials and reduces the risk of secret leakage."

### Q6. What is `DefaultAzureCredential`?

> "It's an Azure Identity credential that provides a common authentication approach across development and Azure environments. Locally it can use developer credentials, while in Azure it can use managed identity."

---

# 25. Senior-Level Scenario

### Interviewer:

> "Your Azure AI application needs to access Azure AI Search, Blob Storage and Key Vault. How would you secure it?"

### Strong answer:

> "I would deploy the application with a managed identity and use Microsoft Entra ID for token-based authentication. I would assign only the required Azure RBAC roles to that identity for Azure AI Search, Blob Storage and Key Vault. I would use `DefaultAzureCredential` in the Python application so that local development can use developer credentials while production uses managed identity. I would follow least privilege and avoid embedding API keys or client secrets in the application."

That is the core **enterprise Azure security pattern** you should be able to explain in the Altimetrik interview.